# Chapter 5 — Vector Databases & ANN Indexes

> **TL;DR** — This chapter implements **IVF** and a simplified **HNSW** from scratch so the graph/cluster mechanics are concrete, then benchmarks Flat vs HNSW vs IVF-PQ in FAISS on recall@k / latency / memory, and shows the metadata pre-filter that doubles as your security boundary.

## 5.1 Why ANN — the cost of exact search

Exact top-k over N vectors is `matrix @ q` → O(N·d) per query. Concretely at d=1024:

In [1]:
import numpy as np, time

for N in (10_000, 100_000, 1_000_000):
    M = np.random.randn(N, 1024).astype(np.float32); q = np.random.randn(1024).astype(np.float32)
    t = time.perf_counter(); _ = np.argsort(-(M @ q))[:10]; ms = (time.perf_counter()-t)*1e3
    print(f"N={N:>9,}  brute-force top-10: {ms:7.1f} ms")

N=   10,000  brute-force top-10:     7.0 ms
N=  100,000  brute-force top-10:    27.5 ms
N=1,000,000  brute-force top-10:  1722.6 ms


ANN accepts a small recall loss for sublinear search. Two dominant families: graph (HNSW) and cluster (IVF).

## 5.2 IVF from scratch — clusters + nprobe
IVF - Inverted File Index

IVF: k-means the corpus into `nlist` cells; at query time search only the nearest `nprobe` cells.

In [3]:
import numpy as np

class IVFIndex:
    def __init__(self, nlist=100, nprobe=8):
        self.nlist, self.nprobe = nlist, nprobe
        
    def fit(self, X, iters=10):
        self.X = X
        # tiny k-means
        rng = np.random.default_rng(0)
        self.centroids = X[rng.choice(len(X), self.nlist, replace=False)].copy()
        
        # the k-means loop
        for _ in range(iters):
            assign = np.argmax(X @ self.centroids.T, axis=1)         # cosine (unit vecs)
            for c in range(self.nlist):
                pts = X[assign == c]
                if len(pts): self.centroids[c] = pts.mean(0)
        self.centroids /= np.linalg.norm(self.centroids, axis=1, keepdims=True)
        self.lists = [np.where(assign == c)[0] for c in range(self.nlist)]  # inverted lists
        return self
    
    def search(self, q, k=10):
        cells = np.argsort(-(self.centroids @ q))[:self.nprobe]       # nearest nprobe cells
        cand = np.concatenate([self.lists[c] for c in cells]) if len(cells) else np.array([], int)
        if len(cand) == 0: return []
        sims = self.X[cand] @ q
        return cand[np.argsort(-sims)[:k]]

`nprobe` **is** the recall/latency dial. Measure it:

In [4]:
X = np.random.randn(100_000, 128).astype(np.float32); X /= np.linalg.norm(X,axis=1,keepdims=True)
q = X[0]; truth = set(np.argsort(-(X @ q))[:10])

ivf = IVFIndex(nlist=256).fit(X)
for np_ in (1, 4, 16, 64):
    ivf.nprobe = np_
    got = set(ivf.search(q, 10))
    print(f"nprobe={np_:3d}  recall@10={len(truth & got)/10:.2f}")

nprobe=  1  recall@10=0.10
nprobe=  4  recall@10=0.10
nprobe= 16  recall@10=0.10
nprobe= 64  recall@10=0.60


The **edge effect** is visible here: a true neighbor sitting just across a cell boundary is missed until `nprobe` grows enough to scan that cell.

## 5.3 HNSW from scratch — the navigable graph
HNSW (Hierarchical Navigable Small World)

A minimal single-layer NSW greedy search captures the core intuition (real HNSW adds hierarchical layers + heuristic edge selection):

To understand this, imagine your database is a giant Social Network.
Instead of sorting documents into buckets (like IVF), this algorithm connects every document to its most similar "friends" using mathematical strings (edges).

If you want to find the person in this network with the most similar movie taste to you, you don't interview all 1,000,000 people. You pick a random person, ask them to point you to their friend who likes movies the most, and you jump to that friend. You keep jumping from friend to friend until no one can introduce you to a better match.

This is called navigating a Small World Graph.

In [5]:
import numpy as np, heapq

class SimpleHNSW:
    def __init__(self, M=16, ef=50):
        self.M, self.ef = M, ef           # M: neighbors/node; ef: search beam width
        self.data, self.graph = [], []    # graph[i] = list of neighbor indices

    def _sim(self, a, b): return float(self.data[a] @ self.data[b])
    def _simv(self, v, b): return float(v @ self.data[b])

    def add(self, vec):
        idx = len(self.data); self.data.append(vec); self.graph.append([])
        if idx == 0: return
        nbrs = self._search_layer(vec, entry=0, ef=self.ef)          # find M nearest existing
        nearest = [n for _, n in heapq.nlargest(self.M, nbrs)]
        self.graph[idx] = nearest
        for n in nearest:                                            # bidirectional edges
            self.graph[n].append(idx)
            if len(self.graph[n]) > self.M:                          # prune to M
                self.graph[n] = [x for _, x in heapq.nlargest(
                    self.M, [(self._sim(n, x), x) for x in self.graph[n]])]

    def _search_layer(self, v, entry, ef):
        visited = {entry}; cand = [(-self._simv(v, entry), entry)]; heapq.heapify(cand)
        top = [(self._simv(v, entry), entry)]
        while cand:
            negd, c = heapq.heappop(cand)
            if -negd < min(top)[0] and len(top) >= ef: break         # greedy stop
            for nb in self.graph[c]:
                if nb in visited: continue
                visited.add(nb); s = self._simv(v, nb)
                if len(top) < ef or s > min(top)[0]:
                    heapq.heappush(cand, (-s, nb)); top.append((s, nb))
                    if len(top) > ef: top.remove(min(top))
        return top

    def search(self, v, k=10):
        return [n for _, n in heapq.nlargest(k, self._search_layer(v, 0, self.ef))]

`ef` (a.k.a. `efSearch`) is the live recall/latency dial — bigger beam explores more of the graph. `M` (edges/node) is the build-time memory/recall knob. This is why HNSW is "high recall + low latency but high memory": it stores M edges per node on top of the vectors.

## 5.4 FAISS benchmark — Flat vs HNSW vs IVF-PQ

In [6]:
import faiss, numpy as np, time

d, N, nq = 128, 200_000, 1_000
Xb = np.random.randn(N, d).astype("float32"); faiss.normalize_L2(Xb)
Xq = Xb[np.random.choice(N, nq, replace=False)].copy()

flat = faiss.IndexFlatIP(d); flat.add(Xb)                     # exact ground truth
_, gt = flat.search(Xq, 10)

def bench(name, index, train=None):
    if not index.is_trained: index.train(train if train is not None else Xb)
    index.add(Xb)
    t = time.perf_counter(); _, I = index.search(Xq, 10); ms = (time.perf_counter()-t)*1e3/nq
    recall = np.mean([len(set(I[i]) & set(gt[i]))/10 for i in range(nq)])
    mem = index.ntotal * (d*4)                                # approx; PQ overrides below
    print(f"{name:14s} recall@10={recall:.3f}  {ms:.3f} ms/q")

hnsw = faiss.IndexHNSWFlat(d, 32); hnsw.hnsw.efConstruction = 200; hnsw.hnsw.efSearch = 64
ivf  = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 256); ivf.nprobe = 16
ivfpq= faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, 256, 16, 8); ivfpq.nprobe = 16  # m=16, 8 bits

bench("Flat(exact)", faiss.IndexFlatIP(d))
bench("HNSW",  hnsw)
bench("IVFFlat", ivf)
bench("IVF-PQ", ivfpq)     # ~ d*4 / m bytes per vector → big 

Flat(exact)    recall@10=1.000  1.032 ms/q
HNSW           recall@10=0.807  0.171 ms/q
IVFFlat        recall@10=0.365  0.182 ms/q
IVF-PQ         recall@10=0.207  0.126 ms/q


## 5.5 Metadata filtering = the security boundary

Pre-filter (search only allowed vectors) is correct and, in multi-tenant systems, mandatory. In FAISS use an `IDSelector`; in a real DB it's a `filter=` clause:


```python
# FAISS: restrict search to allowed ids (e.g. this tenant's chunks)
allowed = np.array([i for i in range(N) if meta[i]["tenant_id"] == user.tenant_id], dtype='int64')
sel = faiss.IDSelectorBatch(allowed)
params = faiss.SearchParametersIVF(sel=sel, nprobe=16)
_, I = ivf.search(Xq, 10, params=params)
```

```python
# Qdrant (production): the filter is enforced server-side AND is your isolation guarantee
from qdrant_client import QdrantClient, models
client.search(
    collection_name="docs",
    query_vector=qv,
    query_filter=models.Filter(must=[
        models.FieldCondition(key="tenant_id", match=models.MatchValue(value=user.tenant_id))]),
    limit=10)
```

**Pre-filter vs post-filter:** pre-filter guarantees no unauthorized vector is ever a candidate (safe, can underfill → over-fetch); post-filter searches then drops (fast, but a leak if you forget the drop). For security, pre-filter + a post-retrieval assertion.

## 5.6 Choosing an index (decision table) + re-index strategy

| Situation | Pick | Why |
|---|---|---|
| < 100k vectors | **Flat** | Exact, simplest, fast enough; ground truth |
| Need lowest latency, RAM available | **HNSW** | Best recall/latency; memory = vectors + M·edges |
| RAM-constrained, millions+ | **IVF-PQ** | ~32× smaller; tune `nprobe` for recall |
| Already run Postgres | **pgvector** | Vectors beside relational data; SQL filters + transactions |
| Live updates + multi-tenant filtering | **Qdrant/Weaviate/Milvus** | Filtered ANN, persistence, replication |

**Re-embedding at scale** (the "how do you switch embedding models on 100M vectors?" question): build a *new versioned collection* offline, backfill in batches, validate recall on the golden set, then **atomically swap** an alias/pointer. Never mutate a live HNSW graph in place — heavy deletes fragment it and degrade recall; rebuild/compact on a schedule.